# Argus prototype data 2: minipixes and calibration records

Each segment is divided into a 64 × 64 grid of **minipixes**, 32 × 32
pixels each. Following one star through full segments is storage and memory intensive, so the
pipeline keeps a small, dedicated time series at selected positions: for each
**active** minipix it stores the minipix's 32 × 32 cutout from every frame at the base observing cadence.
In this archive, the active minipixes are those holding a Gaia DR3 star with
G ≤ 16.

With every cutout the pipeline writes two records:

* a **DQ record**: for each 16 × 16 quadrant of the cutout, the OR of the
  frame's data-quality flags and the number of flagged pixels;
* a **photocal record**: the frame's (versioned) photometric zeropoint at the minipix,
  from the pipeline's spectrophotometric calibration (tied to Gaia XP), with its
  gradient across the minipix and its color term.

Generating a light curve from Argus minipix is done via combining these records with the minipix itself. The sample loaded here holds
every active minipix of segment 391749.

## Parameters

Change these inputs first; leave the calculation cells intact.

In [ ]:
# The target: any star in the staged segment works (see notebook 2 for the list)
TARGET = dict(
    name="Gaia DR3 240827852899491328",
    ra=47.786654, dec=43.945067,  # deg, ICRS
    G=12.727, bp_rp=0.919,        # Gaia DR3 photometry
    gaia_frequency=1.2673242,     # 1/d, gaiadr3.vari_eclipsing_binary
)
SEG = 391749                      # the staged segment holding it

## Setup

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy.visualization import AsinhStretch, ImageNormalize, ZScaleInterval

import argex
from argex import sky, times

warnings.filterwarnings("ignore", category=RuntimeWarning)  # all-NaN slices
GREEK = {"beta": "β", "rho": "ρ"}  # band names in plot labels
plt.rcParams.update({"figure.dpi": 110, "image.origin": "lower"})

# Set ARGUS_SAMPLE to read another copy, such as a download.
SAMPLE = Path(os.environ.get("ARGUS_SAMPLE", "~/s3/schmidt-observatory-system/argus/prototype_v1")).expanduser()
assert (SAMPLE / "manifest.json").is_file(), f"no sample at {SAMPLE}: set ARGUS_SAMPLE"

ra, dec = TARGET["ra"], TARGET["dec"]
FIELD = SAMPLE / f"seg_{SEG}"

## 1. The active minipixes

The map is drawn on the ρ frame from notebook 1, so the minipixes over its
blank side are covered by other pointings.

In [ ]:
active = sorted(int(p.stem.split("_")[1]) for p in (FIELD / "minipix").glob("minipix_*.fits"))
print(len(active), "active minipixes in segment", SEG)

img = argex.read(next((FIELD / "images").glob("rho_*.fits")))
grid, cell = 64, sky.NPIX // 64
px, py = sky.radec_to_pixel(SEG, ra, dec)
target_idx = sky.minipix_index(px, py)
fig = plt.figure(figsize=(6.5, 6.5))
ax = fig.add_subplot(projection=img.wcs)
ax.imshow(img.data, norm=ImageNormalize(img.data, interval=ZScaleInterval()), cmap="gray")
for idx in active:
    x0, y0 = (idx % grid) * cell, (idx // grid) * cell
    mine = idx == target_idx
    ax.add_patch(plt.Rectangle((x0 - 0.5, y0 - 0.5), cell, cell, fill=False,
                               ec="tab:orange" if mine else "tab:cyan", lw=2 if mine else 0.5))
ax.set(xlabel="RA", ylabel="Dec", title=f"Active minipixes (the target's, {target_idx}, in orange)")
plt.show()

## 2. A minipix series

`argex.read()` returns a `MinipixSeries`: the cube of cutouts, each frame's
identity, and the DQ and photocal records. `sky.minipix_index()` gives the
minipix holding a segment pixel.

In [ ]:
series = argex.read(FIELD / "minipix" / f"minipix_{target_idx:04d}.fits")
x, y = series.radec_to_pixel(ra, dec)
print(f"minipix {series.idx}: {len(series)} cutouts, target at ({x:.2f}, {y:.2f})")
print("frames with a DQ record:", series.has_dq.mean(), "  with a photocal record:", series.has_photocal.mean())

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
for ax, band in zip(axes, ("beta", "rho")):
    med = np.nanmedian(series.select(band=band).data, axis=0)
    ax.imshow(med, norm=ImageNormalize(med, stretch=AsinhStretch(0.05)), cmap="gray")
    ax.plot(x, y, "+", color="tab:orange", ms=14)
    ax.axhline(15.5, color="tab:cyan", lw=1.2, ls="--"); ax.axvline(15.5, color="tab:cyan", lw=1.2, ls="--")
    ax.set_title(f"median {GREEK[band]} cutout ({len(series.select(band=band))} frames)")
plt.show()

The dashed lines divide the four DQ quadrants: `[SW, SE, NW, NE]` in that order,
quadrant `q = (y >= 16) * 2 + (x >= 16)`.

## 3. Data quality

The cutouts hold only pixel values, so on their own they cannot tell you
why a point in a light curve is off. The DQ record says whether the pipeline
flagged pixels in that part of the frame: saturation, cosmic rays, satellite
trails, bad calibration pixels, or other localized failure modes. `series.dq(x, y, radius)` ORs the
flags of the quadrants an aperture touches; `argex.dq.names()` spells out
the bits directly.

Our target is well behaved. A bright star elsewhere in the segment shows what
flags look like, and saturates in nearly every frame (some low cloud in one exposure).

In [ ]:
flags = series.dq(x, y, 6)
print("target: frames with any flag under the aperture:", int((flags != 0).sum()), "of", len(series))

bright = argex.read(FIELD / "minipix" / "minipix_1041.fits")  # a G ~ 9 star
bflags = bright.dq()
print("bright star: flagged frames:", int((bflags != 0).sum()), "of", len(bright))
for value in np.unique(bflags[bflags != 0]):
    print(f"  {value:#x}: {argex.dq.names(value)} in {(bflags == value).sum()} frames")
print("flagged pixels per quadrant, first frame:", bright.n_flagged[0], "(of 256)")

med = np.nanmedian(bright.data, axis=0)
fig, ax = plt.subplots(figsize=(3.5, 3.5))
ax.imshow(med, norm=ImageNormalize(med, stretch=AsinhStretch(0.05)), cmap="gray")
ax.axhline(15.5, color="tab:cyan", lw=1.2, ls="--"); ax.axvline(15.5, color="tab:cyan", lw=1.2, ls="--")
for q, (qx, qy) in enumerate([(0, 0), (16, 0), (0, 16), (16, 16)]):
    ax.text(qx + 8, qy + 8, int(np.median(bright.n_flagged[:, q])), color="tab:orange",
            ha="center", va="center", fontsize=12)
ax.set_title("bright star; numbers: median count\nof SATURATED pixels per quadrant", fontsize=9)
plt.show()

The light curves in notebook 3 count a frame as bad when any of
`argex.dq.BAD` is set under the aperture: dead or hot pixels, saturation,
nonlinearity, cosmic rays, satellite trails and bad flat values. It is an
ordinary `IntFlag`, so choose your own set if you prefer (or ask Claude to do it for you. :)).

## 4. Photometric calibration

Each photocal record describes the frame's zeropoint near the minipix as a local
first-order model:

$$
m_\mathrm{AB} = \mathrm{zp} + g_x\,\frac{x - 15.5}{32} + g_y\,\frac{y - 15.5}{32}
+ c\,\big[(g-r) - 0.6\big] - 2.5\log_{10} F
$$

* `zp`: the zeropoint at the minipix center for a flux $F$ in calibrated e⁻/s, for a
  source with $g - r = 0.6$;
* $g_x$, $g_y$: its change across one minipix width (32 pixels);
* $c$: its change per magnitude of $g - r$.

The result is an AB magnitude in the camera's own band (β or ρ). The
zeropoint comes from fitting each frame against Gaia XP spectra integrated
through the band and a model atmosphere (see Garrappa+2024: arXiv:2412.13257 ), so it carries the frame's
transparency and airmass. A frame the pipeline could not calibrate has no
record (NaN here).

In [ ]:
utc = np.asarray(times.to_datetime(series.epochs))
fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)
for band, color in (("beta", "tab:blue"), ("rho", "tab:red")):
    sel = np.array([f.band == band for f in series.frames])
    axes[0].plot(utc[sel], series.zp[sel, 0], ".", ms=2, color=color, label=GREEK[band])
    axes[1].plot(utc[sel], 1e3 * series.grad_x[sel], ".", ms=2, color=color)
    axes[2].plot(utc[sel], series.color[sel], ".", ms=2, color=color)
    
axes[0].set_ylabel("zp (mag)"); axes[0].legend(markerscale=4)
axes[1].set_ylabel("x gradient\n(mmag / 32 px)")
axes[2].set_ylabel("color term\n(mag / mag)")
fig.autofmt_xdate()
plt.show()

Each night's zeropoints drift with airmass. On the night of 2026-10-12
(the morning of the 13th, UTC) they are about 1.5 mag lower in β and 0.9 mag
lower in ρ than on other nights: much less light reached the detector.
Because every frame carries its own zeropoint, calibrated magnitudes stay
comparable across such nights.

In [ ]:
gr = (TARGET["bp_rp"] - 0.355) / 1.198
zp_star = series.zeropoint(x, y, gr=gr)
print(f"target g - r = {gr:.3f}")
print(f"zeropoint at the target, median: {np.nanmedian(zp_star):.3f} mag;"
      f" color correction: {np.nanmedian(series.color * (gr - series.gr_pivot)) * 1e3:+.2f} mmag")

## 5. A per-epoch DQ and zeropoint table

`records()` lays the DQ and photocal records out as one table row per frame:
time, band, frame key, the DQ flags (the OR over the cutout, their names, and
the four quadrants' flags and flagged-pixel counts) and the photocal values.
Given the target's pixel and color, it adds `zp_at`, the zeropoint evaluated
for the target. Saved next to a light curve, it lets you trace any
magnitude back to the records behind it, by frame key.

In [ ]:
epochs = series.records(x, y, gr=gr)
for col in ("zp", "zp_at"):
    epochs[col].format = ".4f"
epochs[["utc", "band", "frame_key", "dq", "dq_names", "n_flagged", "zp", "grad_x", "color", "zp_at"]][:8]

Grouped by night and band, it gives the number of frames, how many were
flagged or uncalibrated, and the range of the zeropoint.

In [ ]:
def night_of(mjd):
    # SRO local noon is about 19:00 UTC: one id per observing night
    return np.floor(np.asarray(mjd) - 0.79).astype(int)

from astropy.table import Table

epochs["night"] = [times.to_iso((n + 0.79 - times.MJDREF) * 86400)[:10] for n in night_of(epochs["mjd"])]
rows = []
for group in epochs.group_by(["night", "band"]).groups:
    zp = group["zp_at"][np.isfinite(group["zp_at"])]
    rows.append(dict(
        night=group["night"][0], band=group["band"][0], frames=len(group),
        flagged=int((group["dq"] != 0).sum()), uncalibrated=int((~group["has_photocal"]).sum()),
        zp_median=np.median(zp) if len(zp) else np.nan,
        zp_min=zp.min() if len(zp) else np.nan, zp_max=zp.max() if len(zp) else np.nan,
    ))
nightly = Table(rows=rows)
for col in ("zp_median", "zp_min", "zp_max"):
    nightly[col].format = ".3f"
epochs.write("epochs_dq_zp.ecsv", overwrite=True)
nightly